In [2]:
import re
import textwrap

import numpy as np
import pandas as pd
import requests
import torch
from bs4 import BeautifulSoup

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)
pd.set_option("display.max_colwidth", 200)

Running on: cpu


## 1. Loading the page

**Scraping the rendered HTML with Beautiful Soup** (what the assignment suggests) gives full control over what to keep, and keeps the structure (headings, lists, infobox) that I want to exploit for chunking.

Wikipedia now rejects requests that don't send a descriptive `User-Agent`, and pages change over time — so I record the **revision ID**. Anyone rerunning this later can load that exact version via `?oldid=<id>` and get comparable results.

In [ ]:
WIKI_URL = "https://en.wikipedia.org/wiki/Apollo_11"
HEADERS = {"User-Agent": "WikiRAG-class-notebook/1.0 (educational use)"}

resp = requests.get(WIKI_URL, headers=HEADERS, timeout=30)
resp.raise_for_status()
soup = BeautifulSoup(resp.text, "html.parser")

page_title = soup.find("h1").get_text(" ", strip=True)
rev = re.search(r'"wgRevisionId":(\d+)', resp.text)
revision_id = rev.group(1) if rev else "unknown"

print(f"Title: {page_title}")
print(f"Revision: {revision_id}  (reproducible URL: {WIKI_URL}?oldid={revision_id})")
print(f"Raw HTML size: {len(resp.text):,} characters")

Title: Apollo 11
Revision: 1371120273  (reproducible URL: https://en.wikipedia.org/wiki/Apollo_11?oldid=1371120273)
Raw HTML size: 1,421,299 characters


## 2. Cleaning: deciding what counts as "the document"

- Removed citation markers
- Removed "References", "Notes", "See also", "External links", "Further reading"
- Removed Navboxes, sidebars, hatnotes, maintenance banners
- RemovedFigures and captions
- Flattened Data tables
- Keep infoboxes

In [ ]:
content = soup.select_one("#mw-content-text .mw-parser-output")

# Strip inline noise first, so it doesn't leak into the infobox or paragraphs.
for sel in ["sup.reference", "span.mw-editsection", "style", "script", ".mw-empty-elt", "#toc", ".toc"]:
    for tag in content.select(sel):
        tag.decompose()

def clean(text: str) -> str:
    text = re.sub(r"\[\s*(\d+|[a-z]|citation needed|note \d+|nb \d+)\s*\]", "", text, flags=re.I)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([.,;:!?)\]])", r"\1", text)
    text = re.sub(r"([(\[])\s+", r"\1", text)
    return text.strip()

def extract_infobox(container) -> str:
    box = container.select_one("table.infobox")
    if box is None:
        return ""
    lines = []
    for row in box.select("tr"):
        th, td = row.find("th"), row.find("td")
        if th and td:
            key, val = clean(th.get_text(" ", strip=True)), clean(td.get_text(" ", strip=True))
            if key and val:
                lines.append(f"{key}: {val}")
    return "\n".join(lines)

infobox_text = extract_infobox(content)

for sel in ["table", "figure", ".thumb", ".gallery", ".navbox", ".sidebar", ".hatnote",
            ".metadata", ".ambox", ".reflist", ".refbegin", ".mw-references-wrap", ".shortdescription"]:
    for tag in content.select(sel):
        tag.decompose()

STOP_SECTIONS = {"see also", "notes", "references", "external links", "further reading",
                 "bibliography", "sources", "citations", "footnotes", "works cited"}

sections = [{"section": "Infobox (key facts)", "text": infobox_text}] if infobox_text else []
h2_title = current = "Introduction"
buffer = []

def flush():
    if buffer:
        sections.append({"section": current, "text": "\n\n".join(buffer)})
        buffer.clear()

for el in content.find_all(["h2", "h3", "h4", "p", "li"]):
    if el.name in ("h2", "h3", "h4"):
        heading = clean(el.get_text(" ", strip=True))
        if el.name == "h2" and heading.lower() in STOP_SECTIONS:
            break                                 
        if el.name == "h2":
            h2_title = current = heading
        else:
            current = f"{h2_title} > {heading}"
        continue
    if el.find_parent("li"):                     
        continue
    text = clean(el.get_text(" ", strip=True))
    if len(text) < 25:
        continue
    buffer.append(f"• {text}" if el.name == "li" else text)
flush()

total_words = sum(len(s["text"].split()) for s in sections)
print(f"{len(sections)} sections, {total_words:,} words of clean text\n")
for s in sections[:12]:
    print(f"  {s['section'][:60]:<60} {len(s['text'].split()):>6} words")

34 sections, 13,525 words of clean text

  Infobox (key facts)                                             278 words
  Introduction                                                    538 words
  Background                                                     1149 words
  Personnel > Prime crew                                          313 words
  Personnel > Backup crew                                         152 words
  Personnel > Support crew                                         88 words
  Personnel > Capsule communicators                                90 words
  Personnel > Flight directors                                      7 words
  Personnel > Other key personnel                                  15 words
  Preparations > Insignia                                         247 words
  Preparations > Call signs                                       146 words
  Preparations > Mementos                                         136 words


In [ ]:
print("=== Infobox (first 15 lines) ===")
print("\n".join(infobox_text.splitlines()[:15]))
print("\n=== Start of the first prose section ===")
first_prose = next(s for s in sections if s["section"] != "Infobox (key facts)")
print(textwrap.fill(first_prose["text"][:1200], 110))

=== Infobox (first 15 lines) ===
Mission type: Crewed lunar landing (G)
Operator: NASA
COSPAR ID: CSM: 1969-059A LM: 1969-059C
SATCAT no.: CSM: 4039 LM: 4041
Mission duration: 8 days, 3 hours, 18 minutes, 35 seconds
Spacecraft: Apollo CSM -107 Apollo LM -5
Manufacturer: CSM: North American Rockwell LM: Grumman
Launch mass: 109,646 lb (49,735 kg)
Landing mass: 10,873 lb (4,932 kg)
Crew size: 3
Members: Neil Armstrong Michael Collins Buzz Aldrin
Callsign: CSM: Columbia LM: Eagle On surface: Tranquility Base
Launch date: July 16, 1969, 13:32:00 (1969-07-16UTC13:32Z) UTC (9:32:00 am EDT)
Rocket: Saturn V SA-506
Launch site: Kennedy, LC ‑ 39A

=== Start of the first prose section ===
Apollo 11 (July 16–24, 1969) was the American spaceflight that first landed humans on the Moon, and the fifth
crewed mission of NASA 's Apollo program. The mission was crewed by Commander Neil Armstrong, Command Module
Pilot Michael Collins, and Lunar Module Pilot Edwin "Buzz" Aldrin, all of whom were on their 

## 3. Chunking

Two decisions that go beyond the reading's default approach:

**1. Section-aware splitting.** I split *each section separately*, so a chunk never straddles two unrelated sections, and I **prefix each chunk's embedding text with the page title and section path** (e.g. `Apollo 11 — Lunar descent > Landing: …`).

**2. Size chosen from the embedding model's limit, not from habit.** `all-MiniLM-L6-v2` has a maximum input of **256 tokens** and *silently truncates* anything longer — the tail of an oversized chunk would simply never be searchable. English averages ~4 characters per token, so I use **800-character chunks with 120 characters of overlap**, which keeps prefix + chunk under the limit.

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def make_chunks(sections, chunk_size=800, chunk_overlap=120, prefix=True):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = []
    for sec in sections:
        for piece in splitter.split_text(sec["text"]):
            chunks.append({
                "id": len(chunks),
                "section": sec["section"],
                "text": piece,
                "embed_text": f"{page_title} — {sec['section']}: {piece}" if prefix else piece,
            })
    return chunks

chunks = make_chunks(sections)
lengths = [len(c["text"]) for c in chunks]
print(f"{len(chunks)} chunks | chars: min {min(lengths)}, median {int(np.median(lengths))}, max {max(lengths)}")

157 chunks | chars: min 43, median 561, max 796


## 4. Embeddings and the vector index

* Embeddings are **L2-normalized**, so the inner product equals cosine similarity.
* I use a **flat (exact) FAISS index**. With a few hundred chunks, brute-force search takes microseconds; approximate indexes (IVF, HNSW) only matter at hundreds of thousands of vectors, and would add tuning parameters and a small accuracy loss for no benefit here.

Building the index is wrapped in a function because Section 8 rebuilds it under different chunking settings to test whether my choices actually matter.

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL, device=device)

tok_lens = [len(embedder.tokenizer(c["embed_text"])["input_ids"]) for c in chunks]
over = sum(l > embedder.max_seq_length for l in tok_lens)
print(f"Embedder max tokens: {embedder.max_seq_length} | longest chunk: {max(tok_lens)} tokens | truncated: {over}/{len(chunks)}")

def build_index(chunks):
    vecs = embedder.encode([c["embed_text"] for c in chunks], batch_size=64,
                           normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    index = faiss.IndexFlatIP(vecs.shape[1])
    index.add(vecs)
    return index

index = build_index(chunks)
print(f"Indexed {index.ntotal} vectors of dimension {index.d}")

def retrieve(query, k=4, chunks_=None, index_=None):
    chunks_ = chunks if chunks_ is None else chunks_
    index_ = index if index_ is None else index_
    q = embedder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, ids = index_.search(q, k)
    return [{**chunks_[i], "score": float(s)} for s, i in zip(scores[0], ids[0]) if i != -1]

def show_hits(hits, width=280):
    for i, h in enumerate(hits, 1):
        print(f"[{i}] score={h['score']:.3f} | {h['section']}")
        print(textwrap.indent(textwrap.shorten(h["text"], width), "     "), "\n")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedder max tokens: 256 | longest chunk: 250 tokens | truncated: 0/157
Indexed 157 vectors of dimension 384


### Testing retrieval *before* adding the LLM

In [8]:
for q in ["What computer alarms happened during the landing?",
          "Which ship picked up the astronauts after splashdown?"]:
    print(f"Q: {q}\n")
    show_hits(retrieve(q, k=3))
    print("-" * 100)

Q: What computer alarms happened during the landing?

[1] score=0.606 | Mission > Lunar descent
     . The extra spurious cycle stealing, as the rendezvous radar updated an involuntary counter, caused the computer alarms. 

[2] score=0.509 | Mission > Lunar descent
     Five minutes into the descent burn, and 6,000 feet (1,800 m) above the surface of the Moon, the LM guidance computer (LGC) distracted the crew with the first of several unexpected 1201 and 1202 program alarms. Inside Mission Control Center, computer engineer Jack Garman [...] 

[3] score=0.497 | Mission > Splashdown and quarantine
     Before dawn on July 24, Hornet launched four Sea King helicopters and three Grumman E-1 Tracers. Two of the E-1s were designated as "air boss" while the third acted as a communications relay aircraft. Two of the Sea Kings carried divers and recovery equipment. The third [...] 

----------------------------------------------------------------------------------------------------
Q: Which sh

## 5. The language model

`Qwen2.5-1.5B-Instruct` is loaded in fp16 on the GPU. I call `generate()` directly with the model's own **chat template**, rather than a high-level pipeline, because instruction-tuned models are trained on a specific role/turn format; ignoring the template is a common reason small models ramble or ignore instructions.

**Greedy decoding (`do_sample=False`)** is deliberate. For a factual Q&A system I want the model's single most likely answer, and I want reruns to be reproducible so the evaluation in Section 8 means something. Sampling adds creativity, which in this setting mostly means hallucination.

In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer

LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
dtype = torch.float16 if device == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
model = AutoModelForCausalLM.from_pretrained(LLM_MODEL).to(device=device, dtype=dtype).eval()

def generate(messages, max_new_tokens=300):
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                           return_tensors="pt", return_dict=True).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             temperature=None, top_p=None, top_k=None,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(generate([{"role": "user", "content": "Reply with exactly: ready"}], max_new_tokens=5))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

ready


## 6. The prompt: making the model stay grounded

The prompt is where "RAG" becomes more than "search + chatbot". Its design choices:

1. **Use only the context - ** Without this, a model that already knows about Apollo 11 will happily answer from memory, and retrieval becomes decoration.
2. Small models hallucinate partly because they're trained to always answer. Giving a specific refusal sentence makes "I don't know" an acceptable response — and a fixed phrase is easy to detect in the evaluation.
3. **Numbered passages with section labels**, so the model can cite `[2]` and a user can check the claim. Citations also nudge the model toward copying facts from passages rather than paraphrasing from memory.
4. **Conversation memory without context bloat.** Past turns are kept as *plain question/answer pairs*, and only the last two exchanges. Re-sending old retrieved passages every turn would fill the small context window quickly and confuse the model about which passages apply to the current question.

**The follow-up problem.** A question like *"Who stayed aboard it?"* is meaningless to the retriever — "it" matches nothing. A full solution uses the LLM to rewrite the follow-up into a standalone question (an extra LLM call per turn). I use a cheap heuristic instead: when there's history, the retrieval query is *previous question + current question*. Section 8 tests whether that's enough.

In [10]:
SYSTEM_PROMPT = (
    f"You answer questions about the Wikipedia article \"{page_title}\". "
    "Use ONLY the numbered context passages provided with each question. "
    "If the answer is not in the passages, reply exactly: \"I couldn't find that in the article.\" "
    "Be concise and cite the passages you used, like [1] or [2]."
)
NOT_FOUND = "couldn't find that in the article"

def build_context(hits):
    return "\n\n".join(f"[{i}] (Section: {h['section']})\n{h['text']}" for i, h in enumerate(hits, 1))

def answer(question, history=None, k=4, use_history_for_retrieval=True):
    history = history or []
    retrieval_query = question
    if history and use_history_for_retrieval:
        last_user = next((m["content"] for m in reversed(history) if m["role"] == "user"), "")
        retrieval_query = f"{last_user} {question}"
    hits = retrieve(retrieval_query, k=k)
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages += history[-4:]
    messages.append({"role": "user",
                     "content": f"Context passages:\n{build_context(hits)}\n\nQuestion: {question}"})
    return generate(messages), hits

def closed_book(question):
    """Same model, no retrieval — the baseline RAG has to beat."""
    return generate([{"role": "system", "content": "Answer concisely from your own knowledge."},
                     {"role": "user", "content": question}])

## 7. Chatting with the page

First, a scripted conversation so the outputs are saved in the notebook, then a live Gradio chat UI.

In [18]:
history = []
for q in ["What was the goal of Apollo 11?",
          "Who were the crew members?",
          "What went wrong during the descent to the Moon?",
          "How did the astronauts get back to Earth?",
          "Who stayed back on the moon?",]:
    reply, hits = answer(q, history)
    history += [{"role": "user", "content": q}, {"role": "assistant", "content": reply}]
    print(f"Question: {q}\nAnswer: {textwrap.fill(reply, 110)}")
    print(f"   sources: {', '.join(dict.fromkeys(h['section'] for h in hits))}\n")

Question: What was the goal of Apollo 11?
Answer: The goal of Apollo 11 was to land humans on the Moon.
   sources: Preparations > First-step decision, Background, Legacy > Cultural significance

Question: Who were the crew members?
Answer: Neil Armstrong, Michael Collins, and Edwin "Buzz" Aldrin.
   sources: Personnel > Flight directors, Personnel > Support crew, Personnel > Other key personnel, Introduction

Question: What went wrong during the descent to the Moon?
Answer: I couldn't find that in the article.
   sources: Introduction, Infobox (key facts), Personnel > Support crew

Question: How did the astronauts get back to Earth?
Answer: The astronauts got back to Earth via splashdown in the Pacific Ocean.
   sources: Introduction, Mission > Lunar surface operations, Background, Mission > Lunar ascent

Question: Who stayed back on the moon?
Answer: Collins stayed back on the moon.
   sources: Introduction, Mission > Lunar surface operations, Background



## Build a chatbot using gradio

In [12]:
import gradio as gr

def _text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return " ".join(p.get("text", "") if isinstance(p, dict) else str(p) for p in content)
    if isinstance(content, dict):
        return content.get("text", "")
    return str(content)

def to_messages(gr_history):
    """Accept either Gradio history format (tuples or role/content dicts)."""
    msgs = []
    for turn in gr_history or []:
        if isinstance(turn, dict):
            msgs.append({"role": turn["role"], "content": _text(turn["content"]).split("\n\n_Sources:")[0]})
        else:
            u, a = turn
            msgs += [{"role": "user", "content": _text(u)},
                     {"role": "assistant", "content": _text(a).split("\n\n_Sources:")[0]}]
    return msgs

def gradio_chat(message, gr_history):
    reply, hits = answer(_text(message), to_messages(gr_history))
    sources = ", ".join(dict.fromkeys(h["section"] for h in hits))
    return f"{reply}\n\n_Sources: {sources}_"

gr.ChatInterface(gradio_chat, title=f"Chat with Wikipedia: {page_title}",
                 description="Answers come only from the article. Sources list the sections used.").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## 8. Conclusions and limitations

### Overall evaluation
The Chatbot gave good responses. It did not halocinate and instead gave a message "I couldn't find that in the article" if it did not know the answer.

### Known limitations (by design)

* **Tables are discarded.** - Any fact that lives only in a table (e.g. some mission timings) is unreachable. A fix would convert each table row to a sentence using its column headers.
* **Single page.**-  The chatbot can't answer questions that need linked articles 
* **Small generator.**  - A 1.5B model is fast and free, but follows instructions less reliably than larger models. 
* **The page changes.** Results are tied to the revision ID printed in Section 1.